In [ ]:
# %% [markdown]
# ## 1. ⚙️ Configuration and Setup

# %%
# --- Configuration Section ---

# File path for the dataset
DATA_FILE_PATH = '/content/Full_1000Data.csv'
# NOTE: Replace with your actual file path.
# You will need to upload this file to your Colab environment.

# Geometry data parameters
GEOMETRY_DIMENSION = (10, 10)
INPUT_SHAPE = GEOMETRY_DIMENSION + (1,) # (10, 10, 1) for CNN input

# S11 output data parameters
S11_OUTPUT_LENGTH = 61

# Data splitting and training parameters
TEST_SPLIT_RATIO = 0.2
RANDOM_SEED = 42
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 1e-4

# Augmentation and Noise parameters
APPLY_AUGMENTATION = True
APPLY_NOISE = True
NOISE_LEVEL = 0.05 # Std deviation for Gaussian noise
AUGMENTATION_FLIPS = True
AUGMENTATION_ROTATIONS = True

# CNN Encoder parameters
ENC_FILTERS = [16, 32, 64]
ENC_KERNEL_SIZE = (3, 3)

# FEDformer Decoder parameters
# Simplified parameters for the attention block
D_MODEL = 64 # Dimension of the model (feature size)
N_HEADS = 8 # Number of attention heads
DROPOUT_RATE = 0.1
N_LAYERS = 2 # Number of encoder/decoder layers

# %%
# --- Install and Import Libraries ---

# Necessary for a basic FEDformer-like structure, beyond standard TensorFlow
# Note: For a *full* FEDformer implementation, a library like PyTorch or
# a custom Keras layer implementation would be required.
# We'll use standard Keras/TensorFlow for the simplified architecture.

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, Flatten, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, Reshape, TimeDistributed, Add, Activation, Conv1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")

# Set random seeds for reproducibility
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

In [ ]:
# %% [markdown]
# ## 2. 🗃️ Data Loading and Preprocessing

# %%
# Load the dataset
try:
    df = pd.read_csv(DATA_FILE_PATH, header=None)
    print(f"Data loaded successfully. Total samples: {len(df)}")
    print(f"Data shape: {df.shape}")
except FileNotFoundError:
    print(f"ERROR: File not found at {DATA_FILE_PATH}. Please upload the file.")
    # Create dummy data for demonstration if file is missing
    print("Creating dummy data for demonstration...")
    N_SAMPLES = 1000
    df = pd.DataFrame(
        np.hstack([
            np.random.randint(0, 2, size=(N_SAMPLES, 100)), # Geometry (100 columns)
            np.random.rand(N_SAMPLES, 61) * 2 - 1          # S11 Real values (61 columns)
        ])
    )

# Extract Input (Geometry) and Output (S11 Real Values)
X_raw = df.iloc[:, :100].values.astype(np.float32)
Y_raw = df.iloc[:, 100:].values.astype(np.float32)

print(f"Input X_raw shape: {X_raw.shape}")
print(f"Output Y_raw shape: {Y_raw.shape}")

# Reshape Input X to be (Samples, 10, 10, 1) for CNN
X = X_raw.reshape(-1, *INPUT_SHAPE)

# Normalize/Scale Output Y (S11 values are typically between -1 and 1 or 0 and 1)
# Assuming S11 real values are already somewhat normalized, but we'll scale them to [-1, 1]
Y = Y_raw
print("NOTE: Assuming S11 data is already scaled, or standard scaling is applied if necessary.")

# Display first sample's geometry and S11
print("\n--- Validation: First Sample Data ---")
print("Geometry (Input X[0,:,:,0]):")
print(X[0,:,:,0])
print(f"S11 Real Values (Output Y[0]): {Y[0][:5]}...{Y[0][-5:]} (Total {Y[0].shape[0]})")

# %%
# --- Optional Augmentation and Noise Injection ---

def apply_augmentation(X_data):
    """Applies random flips and rotations to 10x10 binary geometry."""
    X_aug = []
    for x in X_data:
        x_new = x.copy()
        # Random vertical/horizontal flip
        if AUGMENTATION_FLIPS and np.random.rand() < 0.5:
            x_new = np.flip(x_new, axis=np.random.choice([0, 1]))
        # Random 90-degree rotation
        if AUGMENTATION_ROTATIONS and np.random.rand() < 0.5:
            x_new = np.rot90(x_new, k=np.random.randint(1, 4), axes=(0, 1))

        X_aug.append(x_new)

    return np.array(X_aug)

def apply_noise(X_data):
    """Applies small Gaussian noise to the input geometry (after scaling)."""
    # Noise is applied to the input for robustness, even for binary data.
    noise = np.random.normal(0, NOISE_LEVEL, size=X_data.shape).astype(np.float32)
    X_noisy = X_data + noise
    # Keep values roughly in the binary range after noise
    return np.clip(X_noisy, 0.0, 1.0)


if APPLY_AUGMENTATION:
    X_aug = apply_augmentation(X)
    X = np.concatenate([X, X_aug], axis=0)
    Y = np.concatenate([Y, Y], axis=0)
    print(f"\nAugmentation applied. New total samples: {len(X)}")

if APPLY_NOISE:
    X = apply_noise(X)
    print(f"Noise injection applied to all input samples.")

# Display a validation of noise application
print("--- Validation: Sample 0 (Original) vs Noise/Augmentation ---")
print("Noisy/Augmented Geometry (X[0,:,:,0]):")
print(X[0,:,:,0])

# %%
# Split the data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=TEST_SPLIT_RATIO, random_state=RANDOM_SEED
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")
print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")

In [ ]:
# %% [markdown]
# ## 3. 🧠 Model Architecture (CNN-FEDformer)

# %%
# --- FEDformer Component: Simplified Frequency Enhanced De-composition (FED) Layer ---
# In a full FEDformer, this involves FFT/iFFT and explicit frequency-domain mixing.
# Here, we simulate the decomposition and use a simple time-distributed FFN for
# the 'low-frequency' (trend) and a Transformer block for the 'high-frequency' (seasonal/detail).

class DecompositionLayer(tf.keras.layers.Layer):
    """A simplified layer to decompose the input sequence into Trend and Detail (Seasonal)."""
    def __init__(self, kernel_size, **kwargs):
        super(DecompositionLayer, self).__init__(**kwargs)
        # Use an average pooling or 1D convolution for 'trend' extraction
        self.conv = Conv1D(filters=D_MODEL, kernel_size=kernel_size, padding='same')

    def call(self, inputs):
        # Apply convolution to smooth the sequence (captures low-frequency/trend)
        trend = self.conv(inputs)
        # Detail (High-frequency) is the residual
        detail = inputs - trend
        return trend, detail

# --- FEDformer Component: Attention Block (Modified Transformer Encoder Layer) ---
def fedformer_encoder_layer(input_tensor):
    # 1. Self-Attention (Frequency-Enhanced/Multi-head)
    attn_output = MultiHeadAttention(
        num_heads=N_HEADS, key_dim=D_MODEL, dropout=DROPOUT_RATE
    )(input_tensor, input_tensor)

    # 2. Add & Norm (Residual connection)
    norm1_output = LayerNormalization(epsilon=1e-6)(input_tensor + attn_output)

    # 3. Feed Forward Network (FFN)
    ffn_output = Dense(D_MODEL * 4, activation='relu')(norm1_output)
    ffn_output = Dropout(DROPOUT_RATE)(ffn_output)
    ffn_output = Dense(D_MODEL)(ffn_output)

    # 4. Add & Norm (Residual connection)
    output = LayerNormalization(epsilon=1e-6)(norm1_output + ffn_output)

    return output


# --- CNN Encoder (Feature Extraction) ---
def create_cnn_encoder(input_shape, filters):
    inputs = Input(shape=input_shape, name="Geometry_Input")
    x = inputs

    # Convolutional layers
    for f in filters:
        x = Conv2D(f, ENC_KERNEL_SIZE, activation='relu', padding='same')(x)
        x = Conv2D(f, ENC_KERNEL_SIZE, activation='relu', padding='same', strides=(2, 2))(x)
        x = LayerNormalization(epsilon=1e-6)(x)
        x = Dropout(DROPOUT_RATE)(x)

    # Final Flatten and Dense layer to match D_MODEL
    x = Flatten()(x)

    # Determine the required sequence length for the decoder (e.g., 8 tokens)
    # We will reshape the flattened output to (BATCH_SIZE, SEQUENCE_LEN, D_MODEL)
    SEQUENCE_LEN = 8

    # Project to the total size needed (SEQUENCE_LEN * D_MODEL)
    x = Dense(SEQUENCE_LEN * D_MODEL, activation='relu')(x)

    # Reshape to sequence format (BATCH_SIZE, SEQUENCE_LEN, D_MODEL)
    encoder_output = Reshape((SEQUENCE_LEN, D_MODEL), name="Encoder_Output")(x)

    return Model(inputs=inputs, outputs=encoder_output, name="CNN_Encoder")

# --- FEDformer Decoder (Sequence Prediction) ---
def create_fedformer_decoder(sequence_length, d_model, output_length):
    # Input sequence (from CNN encoder)
    inputs = Input(shape=(sequence_length, d_model), name="Encoder_Sequence_Input")

    x = inputs

    # Apply multiple FEDformer-like Decoder Layers
    for i in range(N_LAYERS):
        # 1. Decomposition
        trend_input, detail_input = DecompositionLayer(kernel_size=3)(x)

        # 2. Trend Processing (Simple FFN/Time-Distributed Dense)
        # Simulates the simple trend block of FEDformer
        trend_output = TimeDistributed(Dense(d_model))(trend_input)

        # 3. Detail Processing (Multi-head Attention)
        # Simulates the attention block of FEDformer
        detail_output = fedformer_encoder_layer(detail_input) # Re-using the structure

        # 4. Recombination and Normalization
        x = trend_output + detail_output
        x = LayerNormalization(epsilon=1e-6)(x) # Final normalization of the block

    # --- Final Output Layer ---
    # Global pooling to capture the entire sequence information
    x = tf.keras.layers.GlobalAveragePooling1D()(x) # (BATCH_SIZE, D_MODEL)

    # Final Dense layer to predict the 61 S11 real values
    output = Dense(output_length, activation='linear', name="S11_Real_Output")(x)

    return Model(inputs=inputs, outputs=output, name="FEDformer_Decoder")

# --- Build the full End-to-End Model ---
cnn_encoder = create_cnn_encoder(INPUT_SHAPE, ENC_FILTERS)
fedformer_decoder = create_fedformer_decoder(
    sequence_length=cnn_encoder.outputs[0].shape[1],
    d_model=D_MODEL,
    output_length=S11_OUTPUT_LENGTH
)

input_geom = Input(shape=INPUT_SHAPE, name="Model_Input")
encoder_out = cnn_encoder(input_geom)
decoder_out = fedformer_decoder(encoder_out)

model = Model(inputs=input_geom, outputs=decoder_out, name="CNN_FEDformer_S11_Predictor")

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='mse', # Mean Squared Error is standard for regression
    metrics=['mae'] # Mean Absolute Error for easier interpretation
)

# %%
# Display Model Summary to validate the architecture
print("--- Validation: CNN Encoder Summary ---")
cnn_encoder.summary()
print("\n--- Validation: FEDformer Decoder Summary ---")
fedformer_decoder.summary()
print("\n--- Validation: Full Model Summary ---")
model.summary()

In [ ]:
# %% [markdown]
# ## 4. 🚀 Model Training

# %%
# Train the model
print(f"Starting training for {EPOCHS} epochs with batch size {BATCH_SIZE}...")
history = model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1, # Use 10% of the training data as validation
    verbose=2 # Show one line per epoch
)

print("\nTraining completed.")

# %%
# --- Validation: Create the Loss Graph ---
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Train Loss (MSE)')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)')
plt.title('Model Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# %% [markdown]
# ## 5. ✅ Model Evaluation

# %%
# Evaluate the model on the test set
print("Evaluating model on test data...")
loss, mae = model.evaluate(X_test, Y_test, verbose=0)
print(f"Test Loss (MSE): {loss:.6f}")
print(f"Test MAE: {mae:.6f}")
print(f"MAE represents the average absolute error in predicting a single S11 real value.")

# %%
# --- Validation: Display prediction vs. true S11 for test samples ---
num_visualize = 3
predictions = model.predict(X_test[:num_visualize], verbose=0)
true_values = Y_test[:num_visualize]

# Create the frequency range for the plot (1 GHz to 6 GHz, 61 points)
frequencies = np.linspace(1.0, 6.0, S11_OUTPUT_LENGTH)

plt.figure(figsize=(15, 5 * num_visualize))
for i in range(num_visualize):
    plt.subplot(num_visualize, 1, i + 1)
    plt.plot(frequencies, true_values[i], label='True S11 (Real)', color='blue')
    plt.plot(frequencies, predictions[i], label='Predicted S11 (Real)', linestyle='--', color='red')
    plt.title(f'Sample {i+1} Prediction vs. True S11 (Real)')
    plt.xlabel('Frequency (GHz)')
    plt.ylabel('S11 Real Value')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# %% [markdown]
# ## 6. 🔮 New Input Inquiry and Prediction

# %%
# --- Function to predict a single new geometry ---
def predict_new_geometry(geometry_array):
    """
    Takes a 10x10 array, reshapes it, and makes a prediction.
    Input must be a 10x10 numpy array or list of 0s and 1s.
    """
    if geometry_array.shape != GEOMETRY_DIMENSION:
        raise ValueError(f"Input geometry must be {GEOMETRY_DIMENSION}")

    # Reshape for model input (1, 10, 10, 1)
    new_input = geometry_array.reshape(1, *INPUT_SHAPE).astype(np.float32)

    # Optional: Apply noise if training used it (for consistency)
    if APPLY_NOISE:
        new_input = apply_noise(new_input)

    # Predict
    prediction = model.predict(new_input, verbose=0)[0]

    return prediction

# %%
# --- Example: Define a new, simple geometry (e.g., a solid patch) ---
# A solid 10x10 patch
new_geometry_1 = np.ones(GEOMETRY_DIMENSION, dtype=np.int32)
new_geometry_1[4:6, 4:6] = 0 # Example: Add a small hole in the center

# Display the input
print("--- Validation: New Geometry Input ---")
print("New Geometry 1 (10x10):")
print(new_geometry_1)

# Predict the S11
predicted_s11_1 = predict_new_geometry(new_geometry_1)

# Plot the result
frequencies = np.linspace(1.0, 6.0, S11_OUTPUT_LENGTH)
plt.figure(figsize=(10, 5))
plt.plot(frequencies, predicted_s11_1, label='Predicted S11 (Real)', color='green')
plt.title('Predicted S11 (Real) for New Geometry 1')
plt.xlabel('Frequency (GHz)')
plt.ylabel('S11 Real Value')
plt.legend()
plt.grid(True)
plt.show()

print("\n--- Prediction Summary ---")
print(f"Predicted S11 (First 5 values): {predicted_s11_1[:5]}")
print(f"Predicted S11 (Last 5 values): {predicted_s11_1[-5:]}")